### Data Selection and Ethical Consideration Note

**Data Selection:**
During the preprocessing phase, features that do not have predictive power and serve only as identifiers (`RowNumber`, `CustomerId`, `Surname`) have been removed from the dataset.

**Ethical Disclaimer:**
Variables such as `Gender` and `Geography` have been retained for the scope of this **experimental project**. However, for a real-world **production deployment**, it is strongly recommended to exclude sensitive demographic data. This precaution is essential to comply with **Ethical AI** standards and to mitigate the risks of algorithmic bias or discrimination based on personal attributes or ethnic origin.


In [1]:
# Import necessary libraries
import pandas as pd 

In [2]:
# Load the processed training data
data = pd.read_csv('../data/processed/train.csv')

In [3]:
# Display the first five rows of the dataset
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,4897,15640464,Parkes,605,France,Male,41,5,91612.91,1,1,1,28427.84,0,0,5,SILVER,529
1,4783,15722611,Cameron,752,France,Female,53,8,114233.18,1,1,1,51587.04,0,0,4,PLATINUM,347
2,1497,15799156,Okwuadigbo,569,Spain,Male,38,8,0.00,2,0,0,79618.79,0,0,1,DIAMOND,225
3,1958,15674922,Beavers,710,France,Male,54,6,171137.62,1,1,1,167023.95,1,1,4,GOLD,451
4,9172,15660475,Ndubueze,411,France,Female,54,9,0.00,1,0,1,76621.49,0,0,1,PLATINUM,739


### Removing Redundant & Risky Features

Based on the **EDA findings**:
*   `RowNumber`, `CustomerId`, `Surname`: Removed as they are non-predictive identifiers.
*   `Complain`: Removed due to **Data Leakage** risk (100% correlation with target).


In [13]:
droped_columns = ['RowNumber', 'CustomerId', 'Surname', 'Complain']
data = data.drop(columns=droped_columns, axis=1)

In [14]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Satisfaction Score,Card Type,Point Earned
0,605,France,Male,41,5,91612.91,1,1,1,28427.84,0,5,SILVER,529
1,752,France,Female,53,8,114233.18,1,1,1,51587.04,0,4,PLATINUM,347
2,569,Spain,Male,38,8,0.00,2,0,0,79618.79,0,1,DIAMOND,225
3,710,France,Male,54,6,171137.62,1,1,1,167023.95,1,4,GOLD,451
4,411,France,Female,54,9,0.00,1,0,1,76621.49,0,1,PLATINUM,739


In [15]:
data.shape

(9000, 14)

### Categorical Encoding

Machine learning models require numerical input. Therefore, categorical variables (`Geography`, `Gender`, `Card Type`) are transformed using **One-Hot Encoding**.
*   **Method:** `pd.get_dummies` with `drop_first=True`.
*   **Reason:** This approach avoids the **Dummy Variable Trap** (multicollinearity) by removing one redundant column for each category.


In [16]:
categorical_cols = ['Geography', 'Gender', 'Card Type']
data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

In [9]:
print(f"Yeni Sütun Sayısı: {data.shape[1]}")
data.head()

Yeni Sütun Sayısı: 17


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Satisfaction Score,Point Earned,Geography_Germany,Geography_Spain,Gender_Male,Card Type_GOLD,Card Type_PLATINUM,Card Type_SILVER
0,605,41,5,91612,1,1,1,28427,0,5,529,0,0,1,0,0,1
1,752,53,8,114233,1,1,1,51587,0,4,347,0,0,0,0,1,0
2,569,38,8,0,2,0,0,79618,0,1,225,0,1,1,0,0,0
3,710,54,6,171137,1,1,1,167023,1,4,451,0,0,1,1,0,0
4,411,54,9,0,1,0,1,76621,0,1,739,0,0,0,0,1,0


### Feature Engineering

To capture deeper patterns and improve model predictive power, new composite features have been derived from existing variables:

*   **`BalanceSalaryRatio`:** Ratio of account balance to estimated salary. Indicates financial stability relative to income.
*   **`TenureByAge`:** Tenure divided by age. Since tenure is a function of age, this normalizes the loyalty metric.
*   **`HasBalance`:** A binary flag (1 or 0) indicating whether the customer has a non-zero balance. This addresses the "zero-inflated" distribution observed in the EDA phase.
*   **`CreditScoreGivenAge`:** Credit score divided by age. Analyzing credit behavior relative to maturity.


In [17]:
print(f"Features Engineering Öncesi Veri : {data.shape[1]}")
data_copy = data.copy()
data_copy['BalanceSalaryRatio'] = data['Balance'] / data['EstimatedSalary']
data_copy['TenureByAge'] = data['Tenure'] / data['Age']
data_copy['HasBalance'] = data['Balance'].apply(lambda x: 1 if x > 0 else 0)
data_copy['CreditScoreGivenAge'] = data['CreditScore'] / data['Age']
data = data_copy
print(f"Features Engineering Sonrası Veri : {data.shape[1]}")
data.head()

Features Engineering Öncesi Veri : 17
Features Engineering Sonrası Veri : 21


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Satisfaction Score,...,Geography_Germany,Geography_Spain,Gender_Male,Card Type_GOLD,Card Type_PLATINUM,Card Type_SILVER,BalanceSalaryRatio,TenureByAge,HasBalance,CreditScoreGivenAge
0,605,41,5,91612.91,1,1,1,28427.84,0,5,...,False,False,True,False,False,True,3.222648,0.121951,1,14.756098
1,752,53,8,114233.18,1,1,1,51587.04,0,4,...,False,False,False,False,True,False,2.214377,0.150943,1,14.188679
2,569,38,8,0.00,2,0,0,79618.79,0,1,...,False,True,True,False,False,False,0.000000,0.210526,0,14.973684
3,710,54,6,171137.62,1,1,1,167023.95,1,4,...,False,False,True,True,False,False,1.024629,0.111111,1,13.148148
4,411,54,9,0.00,1,0,1,76621.49,0,1,...,False,False,False,False,True,False,0.000000,0.166667,0,7.611111
